# Pipeline comparison statistics

This notebook compares the **envelope-only** pipeline with the **envelope + onsets** pipeline.

It performs two mixed-model analyses:

1. **Decoding accuracy**: logistic mixed model  
   `correct ~ pipeline + group + (1 | subject)`

2. **Correlation separation**: linear mixed model on Fisher-z transformed attended–ignored difference  
   `delta_z ~ pipeline + group + (1 | subject)`

The main effect of interest is **pipeline**. The secondary fixed effect is **hearing group**. Subject is included as a random intercept.

In [34]:
from pathlib import Path
import numpy as np
import pandas as pd

# pymer4 is used because it gives lme4-style mixed models from Python.
# If this import fails, run this notebook in the same environment where your GLMM scripts work.
try:
    from pymer4.models import glmer, lmer
except Exception as e:
    raise ImportError(
        "Could not import pymer4. Run this notebook in the environment where pymer4/rpy2/R/lme4 are installed. "
        "Original error: " + repr(e)
    )

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

In [35]:
# --------------------------------------------------
# 1. Path setup
# --------------------------------------------------

# The notebook is intended to live in statistics1/.
# This path logic also works if you run it from the project root.

CWD = Path.cwd()

if CWD.name == "statistics1":
    STATISTICS_DIR = CWD
elif (CWD / "statistics1").exists():
    STATISTICS_DIR = CWD / "statistics1"
else:
    # Fallback: assume current directory is the statistics directory
    STATISTICS_DIR = CWD

ENVELOPE_PATH = STATISTICS_DIR / "aad_trial_level_results.csv"
ENV_ONSET_PATH = STATISTICS_DIR / "envelope_onsets" / "aad_trial_level_results_env_onset.csv"

ACCURACY_REPORT_PATH = STATISTICS_DIR / "pipeline_accuracy_glmm_report.csv"
CORRELATION_REPORT_PATH = STATISTICS_DIR / "pipeline_correlation_lmm_report.csv"

print("Statistics directory:", STATISTICS_DIR)
print("Envelope file:", ENVELOPE_PATH)
print("Envelope + onsets file:", ENV_ONSET_PATH)

Statistics directory: c:\Users\ingeb\OneDrive - Danmarks Tekniske Universitet\Semester 4-Ingeborg\Fagprojekt\AAD_Fagprojekt_v2\statistics1
Envelope file: c:\Users\ingeb\OneDrive - Danmarks Tekniske Universitet\Semester 4-Ingeborg\Fagprojekt\AAD_Fagprojekt_v2\statistics1\aad_trial_level_results.csv
Envelope + onsets file: c:\Users\ingeb\OneDrive - Danmarks Tekniske Universitet\Semester 4-Ingeborg\Fagprojekt\AAD_Fagprojekt_v2\statistics1\envelope_onsets\aad_trial_level_results_env_onset.csv


In [36]:
# --------------------------------------------------
# 2. Load data
# --------------------------------------------------

env = pd.read_csv(ENVELOPE_PATH)
env_onset = pd.read_csv(ENV_ONSET_PATH)

print("Envelope shape:", env.shape)
print("Envelope + onsets shape:", env_onset.shape)

display(env.head())
display(env_onset.head())

# --------------------------------------------------
# 2.1 Fix / restore group labels
# --------------------------------------------------

# In some exported envelope+onsets files, group_HI and hearing_status are empty.
# We therefore use the envelope-only file as the reference source for group labels.

subject_group_map = (
    env[["subject", "group_HI"]]
    .drop_duplicates()
    .set_index("subject")["group_HI"]
)

env_onset["group_HI"] = env_onset["subject"].map(subject_group_map)

# Safety check
missing_group_env = env["group_HI"].isna().sum()
missing_group_env_onset = env_onset["group_HI"].isna().sum()

print("Missing group_HI in envelope file:", missing_group_env)
print("Missing group_HI in envelope+onsets file after mapping:", missing_group_env_onset)

if missing_group_env_onset > 0:
    missing_subjects = env_onset.loc[env_onset["group_HI"].isna(), "subject"].unique()
    raise ValueError(
        "Some subjects in envelope+onsets could not be matched to the envelope file: "
        + ", ".join(missing_subjects)
    )

env["group_HI"] = env["group_HI"].astype(int)
env_onset["group_HI"] = env_onset["group_HI"].astype(int)

Envelope shape: (1408, 13)
Envelope + onsets shape: (1408, 31)


,subject,hearing_status,group_HI,scalp_only,trial_index,r_att,r_ign,r_diff,correct,subject_decoding_accuracy,subject_n_trials,subject_n_correct,summary_path
0,sub-001,hi,1,True,0,0.108249,0.046398,0.061851,1,0.96875,32,31,results_baseline_all\sub-001\sub-001_backward_...
1,sub-001,hi,1,True,1,0.154476,0.016525,0.137951,1,0.96875,32,31,results_baseline_all\sub-001\sub-001_backward_...
2,sub-001,hi,1,True,2,0.071035,-0.007117,0.078152,1,0.96875,32,31,results_baseline_all\sub-001\sub-001_backward_...
3,sub-001,hi,1,True,3,0.117353,0.083574,0.033779,1,0.96875,32,31,results_baseline_all\sub-001\sub-001_backward_...
4,sub-001,hi,1,True,4,0.070426,-0.034507,0.104933,1,0.96875,32,31,results_baseline_all\sub-001\sub-001_backward_...


,subject,hearing_status,group_HI,trial_index,correct,score,r_att_envelope,r_ign_envelope,diff_envelope,r_att_onset,r_ign_onset,diff_onset,r_diff_combined,subject_decoding_accuracy,subject_n_trials,subject_n_features,predictor_names,input_path,tstart,tstop,basis,basis_window,test,partitions,error,selective_stopping,scale_data,score_mode,feature_weights,summary_path,trial_csv_path
0,sub-001,NaN,NaN,0,1,0.075024,0.106912,0.045254,0.061658,0.133078,0.044688,0.088390,0.075024,0.9375,32,2,"envelope,onset",data/processed/env_onset/sub-001_mtrf_env_onse...,-0.5,0.0,0.05,hamming,1,NaN,l2,1,True,mean,"1.0,1.0",results_env_onset\sub-001\sub-001_backward_mtr...,results_env_onset\sub-001\sub-001_backward_mtr...
1,sub-001,NaN,NaN,1,1,0.129550,0.155028,0.015747,0.139281,0.177719,0.057900,0.119819,0.129550,0.9375,32,2,"envelope,onset",data/processed/env_onset/sub-001_mtrf_env_onse...,-0.5,0.0,0.05,hamming,1,NaN,l2,1,True,mean,"1.0,1.0",results_env_onset\sub-001\sub-001_backward_mtr...,results_env_onset\sub-001\sub-001_backward_mtr...
2,sub-001,NaN,NaN,2,1,0.100022,0.070399,-0.008114,0.078513,0.083431,-0.038100,0.121531,0.100022,0.9375,32,2,"envelope,onset",data/processed/env_onset/sub-001_mtrf_env_onse...,-0.5,0.0,0.05,hamming,1,NaN,l2,1,True,mean,"1.0,1.0",results_env_onset\sub-001\sub-001_backward_mtr...,results_env_onset\sub-001\sub-001_backward_mtr...
3,sub-001,NaN,NaN,3,0,-0.018967,0.117675,0.083960,0.033715,0.048953,0.120601,-0.071649,-0.018967,0.9375,32,2,"envelope,onset",data/processed/env_onset/sub-001_mtrf_env_onse...,-0.5,0.0,0.05,hamming,1,NaN,l2,1,True,mean,"1.0,1.0",results_env_onset\sub-001\sub-001_backward_mtr...,results_env_onset\sub-001\sub-001_backward_mtr...
4,sub-001,NaN,NaN,4,1,0.123232,0.068591,-0.031645,0.100237,0.112976,-0.033251,0.146227,0.123232,0.9375,32,2,"envelope,onset",data/processed/env_onset/sub-001_mtrf_env_onse...,-0.5,0.0,0.05,hamming,1,NaN,l2,1,True,mean,"1.0,1.0",results_env_onset\sub-001\sub-001_backward_mtr...,results_env_onset\sub-001\sub-001_backward_mtr...


Missing group_HI in envelope file: 0
Missing group_HI in envelope+onsets file after mapping: 0


In [37]:
# --------------------------------------------------
# 3. Helper functions
# --------------------------------------------------

def group_label_from_group_hi(series: pd.Series) -> pd.Series:
    """Convert group_HI coding to readable group labels."""
    return np.where(series.astype(int) == 1, "HI", "NH")


def fisher_z(r: pd.Series | np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Fisher-z transform with clipping to avoid infinities at exactly +/-1."""
    r = np.asarray(r, dtype=float)
    r = np.clip(r, -1 + eps, 1 - eps)
    return np.arctanh(r)


def clean_pymer4_coefs(coefs: pd.DataFrame, model_type: str) -> pd.DataFrame:
    """Make pymer4 coefficient tables easier to report and save."""
    out = coefs.copy()
    out = out.reset_index().rename(columns={"index": "term"})

    # Normalize common pymer4 column names across versions
    rename_map = {
        "Estimate": "estimate",
        "Est.": "estimate",
        "SE": "se",
        "Std.Err": "se",
        "Z-stat": "z",
        "T-stat": "t",
        "P-val": "p_value",
        "Pr(>|z|)": "p_value",
        "Pr(>|t|)": "p_value",
        "2.5_ci": "ci_low",
        "97.5_ci": "ci_high",
        "CI_lower": "ci_low",
        "CI_upper": "ci_high",
    }
    out = out.rename(columns={c: rename_map.get(c, c) for c in out.columns})

    # Keep useful columns if present
    preferred = ["term", "estimate", "se", "z", "t", "p_value", "ci_low", "ci_high"]
    existing = [c for c in preferred if c in out.columns]
    out = out[existing].copy()

    if model_type == "glmm":
        # Odds ratios are meaningful for logistic models
        if "estimate" in out.columns:
            out["odds_ratio"] = np.exp(out["estimate"])
        if {"ci_low", "ci_high"}.issubset(out.columns):
            out["ci_low_odds_ratio"] = np.exp(out["ci_low"])
            out["ci_high_odds_ratio"] = np.exp(out["ci_high"])

    # Add readable p-value formatting
    if "p_value" in out.columns:
        out["p_value_formatted"] = out["p_value"].apply(
            lambda p: "< .001" if pd.notna(p) and p < 0.001 else (f"{p:.4f}" if pd.notna(p) else "")
        )

    return out

In [38]:
# --------------------------------------------------
# 4. Prepare long-format accuracy data
# --------------------------------------------------

acc_env = env[["subject", "group_HI", "trial_index", "correct"]].copy()
acc_env["pipeline"] = "envelope"

acc_env_onset = env_onset[["subject", "group_HI", "trial_index", "correct"]].copy()
acc_env_onset["pipeline"] = "envelope_onsets"

acc_df = pd.concat([acc_env, acc_env_onset], ignore_index=True)

acc_df["group"] = group_label_from_group_hi(acc_df["group_HI"])
acc_df["correct"] = acc_df["correct"].astype(int)

# Set reference levels through categorical ordering:
# - pipeline reference: envelope
# - group reference: NH
acc_df["pipeline"] = pd.Categorical(acc_df["pipeline"], categories=["envelope", "envelope_onsets"], ordered=False)
acc_df["group"] = pd.Categorical(acc_df["group"], categories=["NH", "HI"], ordered=False)
acc_df["subject"] = acc_df["subject"].astype(str)
acc_df["trial_index"] = acc_df["trial_index"].astype(int)

print("Accuracy long-format shape:", acc_df.shape)
display(acc_df.groupby(["pipeline", "group"])["correct"].agg(["mean", "sum", "count"]))

Accuracy long-format shape: (2816, 6)


mean  sum  count
pipeline        group                      
envelope        NH     0.889205  626    704
                HI     0.923295  650    704
envelope_onsets NH     0.892045  628    704
                HI     0.904830  637    704

In [43]:
# --------------------------------------------------
# 5. Accuracy GLMM fitted directly with R/lme4
# --------------------------------------------------

import numpy as np
import pandas as pd

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects.conversion import localconverter


# --------------------------------------------------
# 5.1 Clean model dataframe
# --------------------------------------------------

acc_df_model = acc_df[["correct", "pipeline", "group", "subject"]].copy()

acc_df_model["correct"] = acc_df_model["correct"].astype(int)
acc_df_model["pipeline"] = acc_df_model["pipeline"].astype(str)
acc_df_model["group"] = acc_df_model["group"].astype(str)
acc_df_model["subject"] = acc_df_model["subject"].astype(str)

acc_df_model = (
    acc_df_model
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .reset_index(drop=True)
)

print(acc_df_model.dtypes)
print(acc_df_model.head())
print("Rows in accuracy model dataframe:", len(acc_df_model))
print("Pipeline levels:", acc_df_model["pipeline"].unique())
print("Group levels:", acc_df_model["group"].unique())


# --------------------------------------------------
# 5.2 Send dataframe to R
# --------------------------------------------------

with localconverter(ro.default_converter + numpy2ri.converter + pandas2ri.converter):
    ro.globalenv["acc_df_r"] = ro.conversion.py2rpy(acc_df_model)


# --------------------------------------------------
# 5.3 Fit GLMM in R
# --------------------------------------------------

ro.r("""
library(lme4)

# Make sure predictors are categorical factors
acc_df_r$pipeline <- factor(acc_df_r$pipeline)
acc_df_r$group <- factor(acc_df_r$group)
acc_df_r$subject <- factor(acc_df_r$subject)

# Optional: print factor levels so interpretation is clear
print(levels(acc_df_r$pipeline))
print(levels(acc_df_r$group))

accuracy_main_model <- glmer(
    correct ~ pipeline + group + (1 | subject),
    data = acc_df_r,
    family = binomial(link = "logit"),
    control = glmerControl(
        optimizer = "bobyqa",
        optCtrl = list(maxfun = 2e5)
    )
)

accuracy_main_summary <- coef(summary(accuracy_main_model))

accuracy_main_report <- data.frame(
    term = rownames(accuracy_main_summary),
    estimate_log_odds = accuracy_main_summary[, "Estimate"],
    SE = accuracy_main_summary[, "Std. Error"],
    z = accuracy_main_summary[, "z value"],
    p_value = accuracy_main_summary[, "Pr(>|z|)"],
    odds_ratio = exp(accuracy_main_summary[, "Estimate"]),
    ci_low_log_odds = accuracy_main_summary[, "Estimate"] - 1.96 * accuracy_main_summary[, "Std. Error"],
    ci_high_log_odds = accuracy_main_summary[, "Estimate"] + 1.96 * accuracy_main_summary[, "Std. Error"]
)

accuracy_main_report$ci_low_odds_ratio <- exp(accuracy_main_report$ci_low_log_odds)
accuracy_main_report$ci_high_odds_ratio <- exp(accuracy_main_report$ci_high_log_odds)

print(summary(accuracy_main_model))
""")


# --------------------------------------------------
# 5.4 Bring report back to Python and save
# --------------------------------------------------

with localconverter(ro.default_converter + numpy2ri.converter + pandas2ri.converter):
    accuracy_report = ro.conversion.rpy2py(ro.r("accuracy_main_report"))

accuracy_report.to_csv(ACCURACY_REPORT_PATH, index=False)

display(accuracy_report)
print("Saved accuracy GLMM report to:", ACCURACY_REPORT_PATH)

correct     int64
pipeline      str
group         str
subject       str
dtype: object
   correct  pipeline group  subject
0        1  envelope    HI  sub-001
1        1  envelope    HI  sub-001
2        1  envelope    HI  sub-001
3        1  envelope    HI  sub-001
4        1  envelope    HI  sub-001
Rows in accuracy model dataframe: 2816
Pipeline levels: <ArrowStringArray>
['envelope', 'envelope_onsets']
Length: 2, dtype: str
Group levels: <ArrowStringArray>
['HI', 'NH']
Length: 2, dtype: str


,term,estimate_log_odds,SE,z,p_value,odds_ratio,ci_low_log_odds,ci_high_log_odds,ci_low_odds_ratio,ci_high_odds_ratio
(Intercept),(Intercept),2.855977,0.270124,10.572823,3.983223e-26,17.391428,2.326534,3.385421,10.242376,29.530430
pipelineenvelope_onsets,pipelineenvelope_onsets,-0.096488,0.130079,-0.741763,4.582307e-01,0.908021,-0.351444,0.158467,0.703671,1.171714
groupNH,groupNH,-0.340325,0.358299,-0.949836,3.421956e-01,0.711539,-1.042591,0.361941,0.352540,1.436114


Saved accuracy GLMM report to: c:\Users\ingeb\OneDrive - Danmarks Tekniske Universitet\Semester 4-Ingeborg\Fagprojekt\AAD_Fagprojekt_v2\statistics1\pipeline_accuracy_glmm_report.csv


In [44]:
# --------------------------------------------------
# 6. Prepare correlation delta data
# --------------------------------------------------

# Envelope-only:
# delta_z = z(r_att) - z(r_ign)

corr_env = env[["subject", "group_HI", "trial_index", "r_att", "r_ign"]].copy()
corr_env["pipeline"] = "envelope"
corr_env["z_att"] = fisher_z(corr_env["r_att"])
corr_env["z_ign"] = fisher_z(corr_env["r_ign"])
corr_env["delta_z"] = corr_env["z_att"] - corr_env["z_ign"]

# Envelope + onsets:
# We Fisher-z transform the envelope and onset correlations separately, then average them.
# This avoids averaging raw Pearson correlations before transformation.

corr_env_onset = env_onset[
    [
        "subject",
        "group_HI",
        "trial_index",
        "r_att_envelope",
        "r_ign_envelope",
        "r_att_onset",
        "r_ign_onset",
    ]
].copy()

corr_env_onset["pipeline"] = "envelope_onsets"

corr_env_onset["z_att"] = np.nanmean(
    np.column_stack([
        fisher_z(corr_env_onset["r_att_envelope"]),
        fisher_z(corr_env_onset["r_att_onset"]),
    ]),
    axis=1
)

corr_env_onset["z_ign"] = np.nanmean(
    np.column_stack([
        fisher_z(corr_env_onset["r_ign_envelope"]),
        fisher_z(corr_env_onset["r_ign_onset"]),
    ]),
    axis=1
)

corr_env_onset["delta_z"] = corr_env_onset["z_att"] - corr_env_onset["z_ign"]

corr_df = pd.concat(
    [
        corr_env[["subject", "group_HI", "trial_index", "pipeline", "z_att", "z_ign", "delta_z"]],
        corr_env_onset[["subject", "group_HI", "trial_index", "pipeline", "z_att", "z_ign", "delta_z"]],
    ],
    ignore_index=True
)

corr_df["group"] = group_label_from_group_hi(corr_df["group_HI"])
corr_df["pipeline"] = pd.Categorical(corr_df["pipeline"], categories=["envelope", "envelope_onsets"], ordered=False)
corr_df["group"] = pd.Categorical(corr_df["group"], categories=["NH", "HI"], ordered=False)
corr_df["subject"] = corr_df["subject"].astype(str)
corr_df["trial_index"] = corr_df["trial_index"].astype(int)

print("Correlation long-format shape:", corr_df.shape)
display(corr_df.groupby(["pipeline", "group"])["delta_z"].agg(["mean", "std", "count"]))

Correlation long-format shape: (2816, 8)


mean       std  count
pipeline        group                           
envelope        NH     0.089596  0.073725    704
                HI     0.092778  0.068056    704
envelope_onsets NH     0.081542  0.068007    704
                HI     0.083986  0.063272    704

In [48]:
# --------------------------------------------------
# 7. Correlation LMM fitted directly with R/lme4
# --------------------------------------------------

import numpy as np
import pandas as pd

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects.conversion import localconverter


# --------------------------------------------------
# 7.1 Clean correlation model dataframe
# --------------------------------------------------

corr_df_model = corr_df[["delta_z", "pipeline", "group", "subject"]].copy()

corr_df_model["delta_z"] = corr_df_model["delta_z"].astype(float)
corr_df_model["pipeline"] = corr_df_model["pipeline"].astype(str)
corr_df_model["group"] = corr_df_model["group"].astype(str)
corr_df_model["subject"] = corr_df_model["subject"].astype(str)

corr_df_model = (
    corr_df_model
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .reset_index(drop=True)
)

print(corr_df_model.dtypes)
print(corr_df_model.head())
print("Rows in correlation model dataframe:", len(corr_df_model))
print("Pipeline levels:", corr_df_model["pipeline"].unique())
print("Group levels:", corr_df_model["group"].unique())


# --------------------------------------------------
# 7.2 Send dataframe to R
# --------------------------------------------------

with localconverter(ro.default_converter + numpy2ri.converter + pandas2ri.converter):
    ro.globalenv["corr_df_r"] = ro.conversion.py2rpy(corr_df_model)


# --------------------------------------------------
# 7.3 Fit LMM in R
# --------------------------------------------------

ro.r("""
library(lme4)
library(lmerTest)

corr_df_r$pipeline <- factor(corr_df_r$pipeline)
corr_df_r$group <- factor(corr_df_r$group)
corr_df_r$subject <- factor(corr_df_r$subject)

print(levels(corr_df_r$pipeline))
print(levels(corr_df_r$group))

correlation_main_model <- lmer(
    delta_z ~ pipeline + group + (1 | subject),
    data = corr_df_r,
    REML = FALSE
)

correlation_main_summary <- coef(summary(correlation_main_model))

correlation_main_report <- data.frame(
    term = rownames(correlation_main_summary),
    estimate = correlation_main_summary[, "Estimate"],
    SE = correlation_main_summary[, "Std. Error"],
    df = correlation_main_summary[, "df"],
    t = correlation_main_summary[, "t value"],
    p_value = correlation_main_summary[, "Pr(>|t|)"],
    ci_low = correlation_main_summary[, "Estimate"] - 1.96 * correlation_main_summary[, "Std. Error"],
    ci_high = correlation_main_summary[, "Estimate"] + 1.96 * correlation_main_summary[, "Std. Error"]
)

print(summary(correlation_main_model))
""")


# --------------------------------------------------
# 7.4 Bring report back to Python and save
# --------------------------------------------------

with localconverter(ro.default_converter + numpy2ri.converter + pandas2ri.converter):
    correlation_report = ro.conversion.rpy2py(ro.r("correlation_main_report"))

correlation_report.to_csv(CORRELATION_REPORT_PATH, index=False)

display(correlation_report)
print("Saved correlation LMM report to:", CORRELATION_REPORT_PATH)

delta_z     float64
pipeline        str
group           str
subject         str
dtype: object
    delta_z  pipeline group  subject
0  0.062244  envelope    HI  sub-001
1  0.139196  envelope    HI  sub-001
2  0.078272  envelope    HI  sub-001
3  0.034127  envelope    HI  sub-001
4  0.105064  envelope    HI  sub-001
Rows in correlation model dataframe: 2816
Pipeline levels: <ArrowStringArray>
['envelope', 'envelope_onsets']
Length: 2, dtype: str
Group levels: <ArrowStringArray>
['HI', 'NH']
Length: 2, dtype: str


,term,estimate,SE,df,t,p_value,ci_low,ci_high
(Intercept),(Intercept),0.092594,0.006479,46.998048,14.291221,9.980790e-19,0.079895,0.105293
pipelineenvelope_onsets,pipelineenvelope_onsets,-0.008423,0.002334,2772.000002,-3.609673,3.120124e-04,-0.012997,-0.003850
groupNH,groupNH,-0.002813,0.009013,43.999994,-0.312080,7.564536e-01,-0.020478,0.014853


Saved correlation LMM report to: c:\Users\ingeb\OneDrive - Danmarks Tekniske Universitet\Semester 4-Ingeborg\Fagprojekt\AAD_Fagprojekt_v2\statistics1\pipeline_correlation_lmm_report.csv


## Does the pipeline effect differ between NH and HI

The models have the framework:

- `pipeline` = main fixed effect
- `group` = secondary fixed effect
- `subject` = random intercept

Interaction models for testing whether the onsets affect the two hearing groups differently:

```python
correct ~ pipeline * group + (1|subject)
delta_z ~ pipeline * group + (1|subject)
```

The interaction term answers whether adding onsets changes performance differently for HI and NH listeners.

In [47]:
# --------------------------------------------------
# 8. Interaction models:
# Does the pipeline effect differ between NH and HI?
# --------------------------------------------------

# These models include:
# - main effect of pipeline
# - main effect of group
# - pipeline × group interaction
# - random subject intercept

ACCURACY_INTERACTION_REPORT_PATH = STATISTICS_DIR / "pipeline_accuracy_glmm_interaction_report.csv"
CORRELATION_INTERACTION_REPORT_PATH = STATISTICS_DIR / "pipeline_correlation_lmm_interaction_report.csv"


# --------------------------------------------------
# 8.1 Accuracy GLMM interaction model fitted directly with R/lme4
# --------------------------------------------------

ro.r("""
accuracy_interaction_model <- glmer(
    correct ~ pipeline * group + (1 | subject),
    data = acc_df_r,
    family = binomial(link = "logit"),
    control = glmerControl(
        optimizer = "bobyqa",
        optCtrl = list(maxfun = 2e5)
    )
)

accuracy_interaction_summary <- coef(summary(accuracy_interaction_model))

accuracy_interaction_report <- data.frame(
    term = rownames(accuracy_interaction_summary),
    estimate_log_odds = accuracy_interaction_summary[, "Estimate"],
    SE = accuracy_interaction_summary[, "Std. Error"],
    z = accuracy_interaction_summary[, "z value"],
    p_value = accuracy_interaction_summary[, "Pr(>|z|)"],
    odds_ratio = exp(accuracy_interaction_summary[, "Estimate"]),
    ci_low_log_odds = accuracy_interaction_summary[, "Estimate"] - 1.96 * accuracy_interaction_summary[, "Std. Error"],
    ci_high_log_odds = accuracy_interaction_summary[, "Estimate"] + 1.96 * accuracy_interaction_summary[, "Std. Error"]
)

accuracy_interaction_report$ci_low_odds_ratio <- exp(accuracy_interaction_report$ci_low_log_odds)
accuracy_interaction_report$ci_high_odds_ratio <- exp(accuracy_interaction_report$ci_high_log_odds)

print(summary(accuracy_interaction_model))
""")

with localconverter(ro.default_converter + numpy2ri.converter + pandas2ri.converter):
    accuracy_interaction_report = ro.conversion.rpy2py(ro.r("accuracy_interaction_report"))

accuracy_interaction_report.to_csv(ACCURACY_INTERACTION_REPORT_PATH, index=False)

display(accuracy_interaction_report)
print("Saved accuracy interaction GLMM report to:", ACCURACY_INTERACTION_REPORT_PATH)

,term,estimate_log_odds,SE,z,p_value,odds_ratio,ci_low_log_odds,ci_high_log_odds,ci_low_odds_ratio,ci_high_odds_ratio
(Intercept),(Intercept),2.940988,0.282849,10.397734,2.538983e-25,18.934540,2.386604,3.495372,10.876494,32.962535
pipelineenvelope_onsets,pipelineenvelope_onsets,-0.255036,0.194729,-1.309699,1.902978e-01,0.774888,-0.636705,0.126632,0.529033,1.135000
groupNH,groupNH,-0.490107,0.384044,-1.276174,2.018941e-01,0.612561,-1.242835,0.262620,0.288565,1.300332
pipelineenvelope_onsets:groupNH,pipelineenvelope_onsets:groupNH,0.286831,0.262024,1.094672,2.736604e-01,1.332199,-0.226737,0.800398,0.797130,2.226427


Saved accuracy interaction GLMM report to: c:\Users\ingeb\OneDrive - Danmarks Tekniske Universitet\Semester 4-Ingeborg\Fagprojekt\AAD_Fagprojekt_v2\statistics1\pipeline_accuracy_glmm_interaction_report.csv


In [49]:
# --------------------------------------------------
# 8.2 Correlation LMM interaction model fitted directly with R/lme4
# --------------------------------------------------

CORRELATION_INTERACTION_REPORT_PATH = STATISTICS_DIR / "pipeline_correlation_lmm_interaction_report.csv"

ro.r("""
correlation_interaction_model <- lmer(
    delta_z ~ pipeline * group + (1 | subject),
    data = corr_df_r,
    REML = FALSE
)

correlation_interaction_summary <- coef(summary(correlation_interaction_model))

correlation_interaction_report <- data.frame(
    term = rownames(correlation_interaction_summary),
    estimate = correlation_interaction_summary[, "Estimate"],
    SE = correlation_interaction_summary[, "Std. Error"],
    df = correlation_interaction_summary[, "df"],
    t = correlation_interaction_summary[, "t value"],
    p_value = correlation_interaction_summary[, "Pr(>|t|)"],
    ci_low = correlation_interaction_summary[, "Estimate"] - 1.96 * correlation_interaction_summary[, "Std. Error"],
    ci_high = correlation_interaction_summary[, "Estimate"] + 1.96 * correlation_interaction_summary[, "Std. Error"]
)

print(summary(correlation_interaction_model))
""")

with localconverter(ro.default_converter + numpy2ri.converter + pandas2ri.converter):
    correlation_interaction_report = ro.conversion.rpy2py(ro.r("correlation_interaction_report"))

correlation_interaction_report.to_csv(CORRELATION_INTERACTION_REPORT_PATH, index=False)

display(correlation_interaction_report)
print("Saved correlation interaction LMM report to:", CORRELATION_INTERACTION_REPORT_PATH)

,term,estimate,SE,df,t,p_value,ci_low,ci_high
(Intercept),(Intercept),0.092778,0.006583,50.093004,14.093041,4.640001e-19,0.079875,0.105682
pipelineenvelope_onsets,pipelineenvelope_onsets,-0.008793,0.003300,2772.000002,-2.664379,7.757739e-03,-0.015261,-0.002325
groupNH,groupNH,-0.003182,0.009310,50.093004,-0.341798,7.339324e-01,-0.021430,0.015066
pipelineenvelope_onsets:groupNH,pipelineenvelope_onsets:groupNH,0.000739,0.004667,2772.000002,0.158312,8.742226e-01,-0.008409,0.009886


Saved correlation interaction LMM report to: c:\Users\ingeb\OneDrive - Danmarks Tekniske Universitet\Semester 4-Ingeborg\Fagprojekt\AAD_Fagprojekt_v2\statistics1\pipeline_correlation_lmm_interaction_report.csv
